# Python Intermedio, primera parte — Sesión 2  

- Selección y filtrado avanzado (`loc`, `iloc`, `query`).
- Agrupaciones y cálculos agregados (`groupby`, `agg`, `transform`).
- Combinación de bases de datos (`merge`, `concat`, `join`).
- Reestructuración de datos (`pivot`, `melt`, `pivot_table`).
- Manejo de valores faltantes, duplicados y creación de variables derivadas.

La idea es que **tú escribas código** en varias celdas en blanco, no solo mirar.  
Si algo falla: ¡mejor! Aprendemos depurando


In [ ]:
import pandas as pd
import numpy as np

# Fijamos una semilla para reproducibilidad
np.random.seed(42)

print(pd.__version__)

---
## 1. Selección y filtrado avanzado (`loc`, `iloc`, `query`)

En Pandas tenemos varias formas de seleccionar y filtrar datos:

- `loc`: selección por **etiquetas** (nombres de filas y columnas).
- `iloc`: selección por **posición** (índices numéricos).
- `query`: filtrado usando una sintaxis similar a una condición lógica sobre el DataFrame.

Vamos a crear un pequeño DataFrame de ejemplo.


### 1.1. Ejemplos básicos de `loc` y `iloc`

- `df.loc[fila, columna]`
- `df.iloc[fila, columna]`

Recordemos que:
- Con `loc` usamos **nombres** o condiciones.
- Con `iloc` usamos **posiciones numéricas** (como en una lista).

### 1.2. Filtrado con condiciones

Podemos combinar condiciones como en SQL o en lógica básica:

- `&` para "y"
- `|` para "o"
- `~` para "no"

Siempre es buena idea usar paréntesis para evitar confusiones.


### 1.3. Uso de `query`

`query` permite escribir filtros usando una sintaxis de tipo expresión:

```python
df.query("ciudad == 'Trujillo' and produccion > 110")
```

Es útil cuando las condiciones se vuelven largas y queremos algo más legible.


### 🧩 Actividad 1 — Selección y filtrado

Usando el DataFrame `df`:

1. Selecciona solo las columnas `ciudad` y `empleo` usando `loc`.
2. Obtén las filas donde `anio` sea 2024 usando `query`.
3. Usando `iloc`, muestra las últimas 3 filas y las columnas de `produccion` y `empleo`.

👉 Escribe tu solución en la celda de abajo. Luego puedes comparar con una posible solución más abajo.


In [ ]:
# Escribe aquí tu solución para la Actividad 1

# 1)


# 2)


# 3)

---
## 2. Agrupaciones y cálculos agregados (`groupby`, `agg`, `transform`)

Muchas veces necesitamos resumir la información por grupo:

- Promedio de producción por ciudad.
- Empleo total por año.
- Crecimiento relativo dentro de cada grupo, etc.

Para eso usamos `groupby` junto con funciones como `mean`, `sum`, `count`, etc.


### 2.1. Uso de `transform`

`transform` devuelve una serie del **mismo tamaño** que el DataFrame original, lo que nos permite crear columnas nuevas que dependen del grupo.

Ejemplo: producción relativa respecto al promedio de su ciudad.


### 🧩 Actividad 2 — Agrupaciones y transformaciones

Con el DataFrame `df`:

1. Calcula el empleo total por ciudad.
2. Crea una nueva columna `empleo_prom_ciudad` con el empleo promedio por ciudad usando `transform`.
3. Crea una columna `empleo_relativo` que sea `empleo / empleo_prom_ciudad`.

👉 Escribe tu solución en la celda de abajo. Luego revisa la propuesta de solución.


In [ ]:
# Escribe aquí tu solución para la Actividad 2

# 1)


# 2)


# 3)

---
## 3. Combinación de bases de datos (`merge`, `concat`, `join`)

En la práctica casi nunca trabajamos con un solo archivo.  
Es común tener:

- Una tabla con producción.
- Otra tabla con precios.
- Otra tabla con empleo, etc.

`merge`, `concat` y `join` nos ayudan a **unir** estas tablas.

Vamos a crear dos DataFrames de ejemplo: producción y precios.


In [ ]:
# DataFrame de produccion por ciudad y anio
df_prod = df[['ciudad', 'anio', 'produccion']].copy()
df_prod

In [ ]:
# Creamos un DataFrame de precios promedio por ciudad y anio
datos_precios = {
    'ciudad': ['Piura', 'Piura', 'Chiclayo', 'Trujillo'],
    'anio': [2023, 2024, 2024, 2023],
    'precio_promedio': [10.5, 11.0, 10.8, 11.2]
}
df_precio = pd.DataFrame(datos_precios)
df_precio

### 3.1. `merge` (similar a un JOIN de SQL)

Podemos hacer:

- `inner` join: intersección.
- `left` join: conserva todas las filas del DataFrame de la izquierda.
- `right`, `outer`, etc.


In [ ]:
# Unimos produccion y precio por ciudad y anio
df_merged = pd.merge(df_prod, df_precio, on=['ciudad', 'anio'], how='left')
df_merged

### 3.2. `concat` para apilar DataFrames

`concat` es útil cuando queremos **apilar** (una debajo de otra) tablas con las mismas columnas.


In [ ]:
# Creamos una copia de df_prod como si fuera otro periodo
df_prod_extra = df_prod.copy()
df_prod_extra['anio'] = df_prod_extra['anio'] + 1  # movemos un año adelante

df_concat = pd.concat([df_prod, df_prod_extra], ignore_index=True)
df_concat

### 🧩 Actividad 3 — Combinación de datos

1. A partir de `df_merged`, crea una columna `valor_produccion` = `produccion * precio_promedio`.
2. Verifica cuántos registros tienen `precio_promedio` nulo (no hubo match en el merge).
3. Usa `concat` para duplicar `df_precio` y luego filtra solo las filas donde `anio` sea mayor a 2023.

👉 Escribe tu solución en la siguiente celda y luego compara.


In [ ]:
# Escribe aquí tu solución para la Actividad 3

# 1)


# 2)


# 3)

---
## 4. Reestructuración de datos (`pivot`, `melt`, `pivot_table`)

A veces necesitamos cambiar el **formato** de nuestros datos:

- Pasar de formato "largo" a "ancho" (y viceversa).
- Crear tablas tipo Excel con filas, columnas y valores agregados.

Para eso usamos:

- `pivot`
- `melt`
- `pivot_table`


In [ ]:
# Tomemos df_merged y quedémonos con ciudad, anio, produccion
df_wide = df_merged[['ciudad', 'anio', 'produccion']].drop_duplicates()
df_wide

### 4.1. `pivot`: de largo a ancho

In [ ]:
# Queremos una tabla donde:
# filas: ciudad
# columnas: anio
# valores: produccion
tabla_pivot = df_wide.pivot(index='ciudad', columns='anio', values='produccion')
tabla_pivot

### 4.2. `melt`: de ancho a largo

In [ ]:
# Volvemos a formato largo
tabla_melt = tabla_pivot.reset_index().melt(id_vars='ciudad', var_name='anio', value_name='produccion')
tabla_melt

### 4.3. `pivot_table`: similar a pivot, pero con agregaciones

Permite usar una función de agregación (por defecto `mean`) y manejar valores repetidos.


In [ ]:
# Ejemplo de pivot_table: promedio de produccion por ciudad y anio
tabla_pt = pd.pivot_table(
    df_merged,
    index='ciudad',
    columns='anio',
    values='produccion',
    aggfunc='mean'
)
tabla_pt

### 🧩 Actividad 4 — Reestructuración

1. Usando `df_merged`, construye una `pivot_table` donde:
   - Las filas sean `anio`.
   - Las columnas sean `ciudad`.
   - Los valores sean `valor_produccion` usando la suma (`sum`).  
2. Convierte esa tabla a formato largo usando `melt`.

👉 Escribe tu solución en la celda de abajo y luego revisa la propuesta.


In [ ]:
# Escribe aquí tu solución para la Actividad 4

# 1)


# 2)

---
## 5. Valores faltantes, duplicados y variables derivadas

En la práctica, los datos reales vienen con:

- Valores faltantes (`NaN`).
- Filas duplicadas.
- Variables que tenemos que construir a partir de otras.

Pandas tiene varias herramientas para esto:
- `isna`, `fillna`, `dropna`
- `duplicated`, `drop_duplicates`
- Creación de columnas con operaciones y `np.where`


In [ ]:
# Introducimos algunos NaN y duplicados a propósito
df_sucio = df_merged.copy()

# Ponemos NaN en algunas celdas
df_sucio.loc[0, 'precio_promedio'] = np.nan
df_sucio.loc[2, 'produccion'] = np.nan

# Duplicamos una fila
df_sucio = pd.concat([df_sucio, df_sucio.iloc[[1]]], ignore_index=True)

df_sucio

### 5.1. Detección y manejo de valores faltantes

In [ ]:
# Contar valores faltantes por columna
df_sucio.isna().sum()

In [ ]:
# Rellenar valores faltantes de produccion con la media de la columna
prod_media = df_sucio['produccion'].mean(skipna=True)
df_sucio['produccion_rellena'] = df_sucio['produccion'].fillna(prod_media)

df_sucio[['produccion', 'produccion_rellena']].head()

### 5.2. Duplicados

In [ ]:
# Detectar filas duplicadas completas
df_sucio.duplicated().sum()

In [ ]:
# Eliminar filas duplicadas
df_sin_duplicados = df_sucio.drop_duplicates()
df_sin_duplicados

### 5.3. Variables derivadas

In [ ]:
# Creamos una variable categórica simple según el nivel de produccion_rellena
df_sin_duplicados['categoria_produccion'] = np.where(
    df_sin_duplicados['produccion_rellena'] >= df_sin_duplicados['produccion_rellena'].median(),
    'Alta',
    'Baja'
)

df_sin_duplicados[['ciudad', 'anio', 'produccion_rellena', 'categoria_produccion']].head()

### 🧩 Actividad 5 — Limpieza y variables derivadas

Trabajando con `df_sucio`:

1. Crea una versión `df_limpio` donde:
   - No haya filas completamente duplicadas.
   - Los valores faltantes en `precio_promedio` se rellenen con el promedio de `precio_promedio`.
2. Crea una columna `precio_alto` que sea:
   - `True` si el precio es mayor al promedio.
   - `False` en caso contrario.
3. Calcula cuántas observaciones tienen `precio_alto == True` por ciudad.

👉 Escribe tu solución en la celda de abajo y luego revisa la propuesta.


In [ ]:
# Escribe aquí tu solución para la Actividad 5

# 1)


# 2)


# 3)